# NLP Daily Challenge: Text Preprocessing, NER, POS Tagging and Word2Vec

In [ ]:
!pip install -q nltk spacy gensim matplotlib pandas scikit-learn
!python -m spacy download en_core_web_sm

In [ ]:
import pandas as pd
import nltk, spacy, string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('tagsets')
nlp=spacy.load("en_core_web_sm")

In [ ]:
data={'Review':[
"At McDonald's the food was ok and the service was bad.",
"I would not recommend this Japanese restaurant to anyone.",
"I loved this restaurant when I traveled to Thailand last summer.",
"The menu of Loving has a wide variety of options.",
"The staff was friendly and helpful at Google's employees restaurant.",
"The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
"I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
"The sushi at Sushi Express is always fresh and flavorful.",
"The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
"The dessert selection at Sweet Treats is to die for!"
]}
df=pd.DataFrame(data)
df

In [ ]:
stop_words=set(stopwords.words('english'))
lemmatizer=WordNetLemmatizer()

def preprocess_text(text):
    tokens=word_tokenize(text.lower())
    tokens=[t for t in tokens if t not in string.punctuation]
    tokens=[t for t in tokens if t not in stop_words]
    tokens=[lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

df["Clean_Review"]=df["Review"].apply(preprocess_text)
display(df[["Review","Clean_Review"]])

In [ ]:
def perform_ner(text):
    doc=nlp(text)
    return [(e.text,e.label_) for e in doc.ents]

def perform_pos_tagging(text):
    return pos_tag(word_tokenize(text))

df["NER_Raw"]=df["Review"].apply(perform_ner)
df["NER_Clean"]=df["Clean_Review"].apply(perform_ner)
df["POS_Raw"]=df["Review"].apply(perform_pos_tagging)
df["POS_Clean"]=df["Clean_Review"].apply(perform_pos_tagging)
print(nltk.help.upenn_tagset("NN"))
display(df[["NER_Raw","NER_Clean","POS_Raw","POS_Clean"]])

In [ ]:
sentences=[txt.split() for txt in df["Clean_Review"]]
model=Word2Vec(sentences=sentences,vector_size=100,window=5,min_count=1,epochs=100)
print("Vocabulary:",len(model.wv))
print("Vector dimension:",model.vector_size)

In [ ]:
def plot_word_embeddings(model):
    words=model.wv.index_to_key
    vectors=model.wv[words]
    coords=PCA(n_components=2).fit_transform(vectors)
    plt.figure(figsize=(12,8))
    plt.scatter(coords[:,0],coords[:,1])
    for i,w in enumerate(words):
        plt.annotate(w,(coords[i,0],coords[i,1]))
    plt.grid()
    plt.show()

plot_word_embeddings(model)

# Analysis

- Word vectors have **100 dimensions**.
- Related words may appear close because Word2Vec learns contextual similarity.
- Since the dataset is very small (10 reviews), embeddings are limited.
- Better results require a much larger corpus and parameter tuning.
